# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chapcoda/flyrank-ML-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# Signal Check 1 - Staleness x visibility

**My Assumptions**
Pages that are stale and still get impresions are strong refresh candidates. For this to hold, I need to see that as days_since_last_update grows, impressions don't collapse. If old pages have essentially zero impressions, they're not worth prioritizing since they're already essentially dead. If old pages still get real search exposure, they're the refresh's main target.

**The bucket table will show:**
* Age bucket (0-30 days, 30-90, 90-180, 180-365, 365+)
* Number of pages in that bucket (n)
* Mean impressions per page in that bucket
* Median impressions per page (median matters more here since impressions are typically skewed)
* % of pages in bucket with less than 100 impressions.

**My rule, in plain words:**

I flag pages for refresh when they're stale enough that they've likely gone unattended, but still visible enough in search that fixing them is worth the effort. A page that's very old and gets no search traffic isn't worth prioritizing — it's already effectively dead. A page updated 31–90 days ago that's still pulling real impressions is exactly the kind of page a content team should catch before it fully decays. Everything else — too fresh to judge, or old and already invisible — gets monitored instead of actively worked.

**Reason codes:**

| reason_code | condition | action |
|---|---|---|
| `STALE-VISIBLE` | age 31–90 days AND impressions ≥ 100 | `refresh` |
| `AGING-LOW-VISIBILITY` | age > 90 days AND impressions < 100 | `monitor` |
| `TOO-FRESH` | age ≤ 30 days | `monitor` |
| `STANDARD` | everything else | `monitor` |

*Note: an earlier version of this rule also planned a CTR-vs-position signal (`CTR-BELOW-TIER-EXPECTED` → `refresh_and_review_ctr`). That signal was investigated and dropped — see the Signal Check 2 verdict below — so the rule here runs on staleness × visibility alone.*

In [1]:
# ============ SETUP (self-contained — safe to re-run any time) ============
import os, sys, subprocess

# Repo setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# Install DuckDB + huggingface if needed
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

# Login to Hugging Face
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    HF_TOKEN = getpass("HF token: ")
login(token=HF_TOKEN)

# DuckDB connection
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN \'{HF_TOKEN}\'
);
""")

# Path variables
DATASET = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = f"{DATASET}/dim_content.parquet"

print("Setup complete.")
print("Working dir:", os.getcwd())

# ============ SIGNAL CHECK 1 — STALENESS × VISIBILITY ============
q_staleness = f"""
WITH page_impressions AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS total_impressions
    FROM read_parquet(\'{FACT_MARCH}\')
    GROUP BY content_hash_id
),
page_staleness AS (
    SELECT
        content_hash_id,
        DATE_DIFF(\'day\', content_updated_date, DATE \'2026-03-31\') AS days_since_update
    FROM read_parquet(\'{DIM_CONTENT}\')
    WHERE content_updated_date IS NOT NULL
      AND is_published = TRUE
      AND is_deleted = FALSE
),
joined AS (
    SELECT
        s.days_since_update,
        COALESCE(p.total_impressions, 0) AS impressions,
        CASE
            WHEN s.days_since_update <= 30  THEN \'1) 0-30d\'
            WHEN s.days_since_update <= 90  THEN \'2) 31-90d\'
            WHEN s.days_since_update <= 180 THEN \'3) 91-180d\'
            WHEN s.days_since_update <= 365 THEN \'4) 181-365d\'
            ELSE                                 \'5) 365+d\'
        END AS age_bucket
    FROM page_staleness s
    LEFT JOIN page_impressions p ON s.content_hash_id = p.content_hash_id
    WHERE s.days_since_update >= 0
)
SELECT
    age_bucket,
    COUNT(*) AS n,
    ROUND(AVG(impressions), 1) AS mean_impressions,
    MEDIAN(impressions) AS median_impressions,
    ROUND(100.0 * SUM(CASE WHEN impressions >= 100 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_visible_100plus
FROM joined
GROUP BY age_bucket
ORDER BY age_bucket
"""

display(con.execute(q_staleness).fetchdf())

# ============ SIGNAL CHECK 2 — CTR vs POSITION TIER (fixed) ============
q_ctr_position = f"""
WITH page_march AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_clicks ELSE 0 END) AS clicks,
        AVG(CASE WHEN gsc_data_available = TRUE THEN gsc_avg_position END) AS avg_position
    FROM read_parquet(\'{FACT_MARCH}\')
    GROUP BY content_hash_id
    HAVING SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) >= 100
),
tiered AS (
    SELECT
        content_hash_id,
        impressions,
        clicks,
        avg_position,
        ROUND(clicks * 1.0 / impressions, 4) AS ctr,
        CASE
            WHEN avg_position <= 1  THEN \'1) page_1\'
            WHEN avg_position <= 3  THEN \'2) top_3\'
            WHEN avg_position <= 10 THEN \'3) striking\'
            WHEN avg_position <= 20 THEN \'4) page_2\'
            ELSE                          \'5) deep\'
        END AS position_tier
    FROM page_march
    WHERE avg_position IS NOT NULL
),
overall AS (
    SELECT AVG(ctr) AS overall_mean_ctr FROM tiered
)
SELECT
    t.position_tier,
    COUNT(*) AS n,
    ROUND(AVG(t.ctr), 4) AS mean_ctr,
    ROUND(STDDEV(t.ctr), 4) AS stddev_ctr,
    ROUND(MEDIAN(t.ctr), 4) AS median_ctr,
    ROUND(100.0 * SUM(CASE WHEN t.ctr < o.overall_mean_ctr * 0.5 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_below_half_overall_mean
FROM tiered t
CROSS JOIN overall o
GROUP BY t.position_tier, o.overall_mean_ctr
ORDER BY t.position_tier
"""

con.execute(q_ctr_position).fetchdf()

Setup complete.
Working dir: /content/flyrank-ml-internship-starter


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,mean_impressions,median_impressions,pct_visible_100plus
0,1) 0-30d,973,95.2,3.0,9.8
1,2) 31-90d,34565,980.3,115.0,51.6
2,3) 91-180d,3582,131.6,0.0,3.4
3,4) 181-365d,3549,4.3,0.0,0.4


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_tier,n,mean_ctr,stddev_ctr,median_ctr,pct_below_half_overall_mean
0,1) page_1,446,0.0018,0.0032,0.0006,61.7
1,2) top_3,8585,0.0037,0.0045,0.0025,32.4
2,3) striking,46864,0.0032,0.0047,0.0019,41.6
3,4) page_2,21474,0.0024,0.0037,0.0010,54.7
4,5) deep,24072,0.0012,0.0029,0.0000,74.7


In [2]:
# ============ DIAGNOSTIC: raw clicks/impressions for page_1 ============
q_diagnostic = f"""
WITH page_march AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_clicks ELSE 0 END) AS clicks,
        AVG(CASE WHEN gsc_data_available = TRUE THEN gsc_avg_position END) AS avg_position
    FROM read_parquet(\'{FACT_MARCH}\')
    GROUP BY content_hash_id
    HAVING SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) >= 100
)
SELECT content_hash_id, impressions, clicks, avg_position,
       ROUND(clicks * 1.0 / impressions, 4) AS ctr
FROM page_march
WHERE avg_position <= 1
ORDER BY impressions DESC
LIMIT 15
"""

con.execute(q_diagnostic).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,avg_position,ctr
0,content_a2ebfe89d44ed4ff,35426.0,46.0,0.596619,0.0013
1,content_0e7bc6744950f433,34700.0,70.0,0.761700,0.0020
2,content_1eb85de9c101e25b,28753.0,82.0,0.544579,0.0029
3,content_a5e1a1f3b30597bf,25429.0,19.0,0.952650,0.0007
4,content_a3491b6328a1816f,16005.0,29.0,0.750536,0.0018
5,content_f4895f580257c3b2,14682.0,13.0,0.348125,0.0009
6,content_bd84587a618b32ec,11370.0,43.0,0.907089,0.0038
7,content_35ddcd760c64da18,11090.0,1.0,0.989128,0.0001
8,content_ba2e51485ee663fa,10969.0,25.0,0.999024,0.0023
9,content_d1a3abe6dca046b6,10525.0,20.0,0.888416,0.0019


# Signal 2 CTR x Position

**Verdict: FALSE**

Ran a diagnostic on the 15 highest-impression pages in the `page_1` tier (avg_position ≤ 1) to check
whether the CTR anomaly was small-sample noise or a real problem. Two things ruled out noise:

1. `avg_position` values came back fractional (0.34–0.99) across every row. Real search-engine
   position is never below 1 — position 1 is the best possible rank. Values under 1.0 don't
   correspond to any real SERP position, which means this field isn't literal search rank the way
   the rule assumed.
2. CTR stayed near-zero (0.07%–0.4%) on pages with 9,000–35,000 impressions — the top-traffic pages
   in the theoretically best position tier. Real position-1 CTR is typically 20–40%. This held
   across all 15 rows, not just a few, so it isn't sampling noise from the small tier size (n=446).

The math itself is correct (clicks / impressions matches the reported ctr), so this isn't a query
bug. Most likely explanation: this is a synthetic training dataset, and the data generator didn't
model a realistic CTR-vs-position relationship — so `avg_position` and CTR here don't behave like
real-world search performance data.

**Decision:** dropped CTR-vs-position as a scoring signal. The `CTR-BELOW-TIER-EXPECTED` reason code
is removed. The baseline rule below relies on staleness × visibility only.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# ============ SECTION 2 — BUILD THE RANKED QUEUE ============
# Signal used: staleness x visibility only (CTR-vs-position dropped — see Signal Check 2 verdict: FALSE,
# raw diagnostic showed avg_position values outside the real 1-100 SERP range and CTR uniformly
# near-zero regardless of rank, indicating the field doesn't reflect real search-position behavior.)

q_baseline = f"""
WITH page_impressions AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS impressions
    FROM read_parquet(\'{FACT_MARCH}\')
    GROUP BY content_hash_id
),
page_staleness AS (
    SELECT
        content_hash_id,
        content_type,
        DATE_DIFF(\'day\', content_updated_date, DATE \'2026-03-31\') AS days_since_update
    FROM read_parquet(\'{DIM_CONTENT}\')
    WHERE content_updated_date IS NOT NULL
      AND is_published = TRUE
      AND is_deleted = FALSE
)
SELECT
    s.content_hash_id,
    s.content_type,
    s.days_since_update,
    COALESCE(p.impressions, 0) AS impressions,
    CASE
        WHEN s.days_since_update <= 30  THEN \'1) 0-30d\'
        WHEN s.days_since_update <= 90  THEN \'2) 31-90d\'
        WHEN s.days_since_update <= 180 THEN \'3) 91-180d\'
        WHEN s.days_since_update <= 365 THEN \'4) 181-365d\'
        ELSE                                 \'5) 365+d\'
    END AS age_bucket
FROM page_staleness s
LEFT JOIN page_impressions p ON s.content_hash_id = p.content_hash_id
WHERE s.days_since_update >= 0
"""

df = con.execute(q_baseline).fetchdf()

# --- staleness weight per bucket, taken directly from Signal Check 1\'s own measured
#     pct_visible_100plus values (9.8 / 51.6 / 3.4 / 0.4), normalized to 0-1 ---
bucket_weights = {
    "1) 0-30d": 9.8 / 51.6,
    "2) 31-90d": 51.6 / 51.6,    # empirically strongest bucket
    "3) 91-180d": 3.4 / 51.6,
    "4) 181-365d": 0.4 / 51.6,
    "5) 365+d": 0.1 / 51.6,      # not directly measured, treated as floor
}
df["staleness_weight"] = df["age_bucket"].map(bucket_weights)

# --- combine staleness weight with visibility (log-dampened so a few huge-impression
#     pages don\'t dominate the whole ranking) ---
import numpy as np
df["action_score"] = df["staleness_weight"] * np.log1p(df["impressions"])

# --- reason code + action, per the revised table above ---
def assign_reason(row):
    if row["age_bucket"] == "2) 31-90d" and row["impressions"] >= 100:
        return "STALE-VISIBLE", "refresh"
    elif row["days_since_update"] > 90 and row["impressions"] < 100:
        return "AGING-LOW-VISIBILITY", "monitor"
    elif row["days_since_update"] <= 30:
        return "TOO-FRESH", "monitor"
    else:
        return "STANDARD", "monitor"

df[["reason_code", "action"]] = df.apply(lambda r: pd.Series(assign_reason(r)), axis=1)

# --- rank and write ---
df = df.sort_values("action_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

output_cols = ["rank", "content_hash_id", "content_type", "days_since_update",
               "age_bucket", "impressions", "staleness_weight", "action_score",
               "reason_code", "action"]

os.makedirs("work/outputs", exist_ok=True)
df[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(df)} rows to work/outputs/baseline_action_score.csv")
df[output_cols].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 42669 rows to work/outputs/baseline_action_score.csv


,rank,content_hash_id,content_type,days_since_update,age_bucket,impressions,staleness_weight,action_score,reason_code,action
0,1,content_f43118e089ecc69a,keyword article,34,2) 31-90d,139417.0,1.0,11.845232,STALE-VISIBLE,refresh
1,2,content_9c057b66c30a3abb,keyword article,34,2) 31-90d,83834.0,1.0,11.336606,STALE-VISIBLE,refresh
2,3,content_73aa61dcedebbf30,keyword article,34,2) 31-90d,80124.0,1.0,11.291343,STALE-VISIBLE,refresh
3,4,content_80eb6221de550658,keyword article,34,2) 31-90d,79766.0,1.0,11.286865,STALE-VISIBLE,refresh
4,5,content_ac7b77e81c53d636,keyword article,34,2) 31-90d,79651.0,1.0,11.285422,STALE-VISIBLE,refresh
5,6,content_49267c758cdcb3a8,keyword article,34,2) 31-90d,73768.0,1.0,11.208694,STALE-VISIBLE,refresh
6,7,content_e9f2d0579387d3c3,keyword article,34,2) 31-90d,73503.0,1.0,11.205095,STALE-VISIBLE,refresh
7,8,content_57dcb96896f9a33c,keyword article,34,2) 31-90d,70026.0,1.0,11.156636,STALE-VISIBLE,refresh
8,9,content_b875a2f306635a58,keyword article,34,2) 31-90d,68657.0,1.0,11.136893,STALE-VISIBLE,refresh
9,10,content_f2388a4b87a3b1dc,keyword article,34,2) 31-90d,67646.0,1.0,11.122058,STALE-VISIBLE,refresh


In [4]:
q_check = f"""
SELECT days_since_update, content_type, COUNT(*) AS n
FROM ({q_baseline})
WHERE age_bucket = \'2) 31-90d\'
GROUP BY days_since_update, content_type
ORDER BY n DESC
LIMIT 10
"""
con.execute(q_check).fetchdf()

,days_since_update,content_type,n
0,34,keyword article,30216
1,34,feedly article,1828
2,33,keyword article,497
3,74,keyword article,458
4,81,keyword article,381
5,36,keyword article,357
6,32,keyword article,353
7,34,comparison article,243
8,35,keyword article,173
9,35,feedly article,22


**Data artifact found:** 87% of pages in the 31–90 day bucket (30,216 of ~34,565) share the exact
same `days_since_update = 34`. This is far too concentrated to reflect independent editorial
activity — it's more consistent with a bulk system event (migration/seed date) than real per-page
updates. Consequence: within this bucket, `days_since_update` doesn't meaningfully discriminate
between pages — `action_score` ranking here is effectively driven by `impressions` alone, with
staleness acting as a bucket-level gate rather than a fine-grained signal. This doesn't invalidate
the rule (visibility is still a legitimate urgency signal), but the "staleness" framing is weaker
than the bucket-level Signal Check 1 findings implied.

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

All 10 top picks share the same `reason_code` (`STALE-VISIBLE`) and `action` (`refresh`), and all
sit in the `days_since_update = 34` cluster documented earlier as a likely batch update artifact.
Within that tied group, the ranking is driven entirely by `impressions` — so what differentiates
rank 1 from rank 10 here is visibility, not staleness.

| rank | content_hash_id | impressions | action | reason_code | confidence note |
|---|---|---|---|---|---|
| 1 | content_f43118e089ecc69a | 139,417 | refresh | STALE-VISIBLE | High confidence in visibility (largest impression count in the set); staleness figure itself is less trustworthy given the shared 34-day cluster. |
| 2 | content_9c057b66c30a3abb | 83,834 | refresh | STALE-VISIBLE | Same as above — strong visibility signal, same staleness caveat. |
| 3 | content_73aa61dcedebbf30 | 80,124 | refresh | STALE-VISIBLE | Impressions still well above the bucket's own median (115), so visibility is real. |
| 4 | content_80eb6221de550658 | 79,766 | refresh | STALE-VISIBLE | Same pattern. |
| 5 | content_ac7b77e81c53d636 | 79,651 | refresh | STALE-VISIBLE | Same pattern. |
| 6 | content_49267c758cdcb3a8 | 73,768 | refresh | STALE-VISIBLE | Same pattern. |
| 7 | content_e9f2d0579387d3c3 | 73,503 | refresh | STALE-VISIBLE | Same pattern. |
| 8 | content_57dcb96896f9a33c | 70,026 | refresh | STALE-VISIBLE | Same pattern. |
| 9 | content_b875a2f306635a58 | 68,657 | refresh | STALE-VISIBLE | Same pattern. |
| 10 | content_f2388a4b87a3b1dc | 67,646 | refresh | STALE-VISIBLE | Same pattern. |

**What would make any of these wrong:** the shared 34-day `days_since_update` most likely reflects
a system-level event (migration/seed) rather than these specific pages being independently neglected
by an editor — so if that date turns out to mean something other than "last content edit" (e.g. a
metadata refresh, a re-crawl timestamp, or a warehouse load date), the staleness half of the rule
doesn't actually apply to these pages and `refresh` would be the wrong action regardless of how
correct the impressions signal is. Separately, a single-month (March) impression count can't
distinguish a page that's steadily strong from one that's about to decline — no trend/future-window
data was used here by design (leakage-safe), so "refresh" is a decision-support suggestion based on
current visibility, not a prediction that these specific pages are declining.

In [5]:
top10 = pd.read_csv("work/outputs/baseline_action_score.csv").head(10)
top10

,rank,content_hash_id,content_type,days_since_update,age_bucket,impressions,staleness_weight,action_score,reason_code,action
0,1,content_f43118e089ecc69a,keyword article,34,2) 31-90d,139417.0,1.0,11.845232,STALE-VISIBLE,refresh
1,2,content_9c057b66c30a3abb,keyword article,34,2) 31-90d,83834.0,1.0,11.336606,STALE-VISIBLE,refresh
2,3,content_73aa61dcedebbf30,keyword article,34,2) 31-90d,80124.0,1.0,11.291343,STALE-VISIBLE,refresh
3,4,content_80eb6221de550658,keyword article,34,2) 31-90d,79766.0,1.0,11.286865,STALE-VISIBLE,refresh
4,5,content_ac7b77e81c53d636,keyword article,34,2) 31-90d,79651.0,1.0,11.285422,STALE-VISIBLE,refresh
5,6,content_49267c758cdcb3a8,keyword article,34,2) 31-90d,73768.0,1.0,11.208694,STALE-VISIBLE,refresh
6,7,content_e9f2d0579387d3c3,keyword article,34,2) 31-90d,73503.0,1.0,11.205095,STALE-VISIBLE,refresh
7,8,content_57dcb96896f9a33c,keyword article,34,2) 31-90d,70026.0,1.0,11.156636,STALE-VISIBLE,refresh
8,9,content_b875a2f306635a58,keyword article,34,2) 31-90d,68657.0,1.0,11.136893,STALE-VISIBLE,refresh
9,10,content_f2388a4b87a3b1dc,keyword article,34,2) 31-90d,67646.0,1.0,11.122058,STALE-VISIBLE,refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks found:**

Checked two likely failure modes in the rule:

1. *Pages just past the 90-day cutoff that are still highly visible.* None found — no page with
   91–110 days since update and ≥500 impressions exists in this data. This is reassuring for the
   current snapshot (the hard 90-day cutoff isn't currently costing any high-visibility pages), but
   it doesn't mean the cutoff itself is well-justified — it's still an arbitrary line inherited from
   Signal Check 1's bucket boundaries, and a future data pull could easily produce cases where it
   matters.

2. *Pages flagged `TOO-FRESH` (≤30 days) that already have real visibility.* Found 5 clear examples —
   e.g. `content_65b8a4998e633d89` at 22 days old with 8,905 impressions, and `content_d8ca9e67dbeeeb50`
   at 26 days with 8,238 impressions. Both already exceed the *median* impression count of the
   best-performing bucket in Signal Check 1 (115 impressions, 31-90d bucket). The rule's assumption —
   that fresh pages don't have enough signal yet — doesn't hold for these: they're being marked
   `monitor` purely on age, when their visibility already suggests they're worth watching more closely.
   This is a genuine weak pick, not a data artifact: the fix would be adding an impression floor that
   can override `TOO-FRESH` regardless of age, rather than treating age as an absolute gate.

**Leakage check:**

Every column touched by the scoring pipeline, confirmed against the data contract from `w03`:

- Feature window: March 2026 only (`fact_content_daily_performance`, `month=2026-03` partition).
- From the fact table: `gsc_data_available`, `gsc_impressions` only.
- From `dim_content`: `content_hash_id`, `content_type`, `content_updated_date`, `is_published`, `is_deleted`.
- Not used anywhere: `gsc_clicks`, `gsc_avg_position` (dropped with the CTR signal — see Signal Check 2
  verdict), any `ga4_*` column, `trend_direction`, `trend_pct`, `is_declining_label` (label-derived —
  would leak the target), and nothing from the May 2026 target window.

No product flags, client identifiers, or future-window data appear anywhere in this pipeline. The
score is built entirely from March 2026 observed data, consistent with the leakage-safe feature
window defined in `w03_data_contract.ipynb`.

In [6]:
df_full = pd.read_csv("work/outputs/baseline_action_score.csv")

# Weak pick check 1: pages just past the 90-day cutoff that still have strong visibility —
# these get bucketed into a lower-priority reason_code purely because they missed the
# 31-90 day window by a little, which exposes how arbitrary a hard cutoff can be.
edge_cases = df_full[
    (df_full["days_since_update"].between(91, 110)) & (df_full["impressions"] >= 500)
].sort_values("impressions", ascending=False).head(5)

# Weak pick check 2: pages marked TOO-FRESH (<=30 days) that already have unusually high
# impressions — the rule assumes fresh pages don\'t have enough signal yet, but a high-traffic
# page even at 20-30 days old might already be worth watching.
fresh_high_vis = df_full[
    (df_full["reason_code"] == "TOO-FRESH") & (df_full["impressions"] >= 500)
].sort_values("impressions", ascending=False).head(5)

print("Edge cases: just past the 90-day cutoff, still highly visible")
display(edge_cases)

print("\nEdge cases: flagged TOO-FRESH but already highly visible")
display(fresh_high_vis)

# --- Leakage check: explicitly list every column the pipeline touches ---
print("\n=== LEAKAGE CHECK ===")
print("Feature window used: March 2026 only (fact_content_daily_performance, month=2026-03)")
print("Columns used from fact table: gsc_data_available, gsc_impressions")
print("Columns used from dim_content: content_hash_id, content_type, content_updated_date, is_published, is_deleted")
print("\nColumns explicitly NOT used anywhere in this pipeline:")
print(" - gsc_clicks, gsc_avg_position (dropped with the CTR signal)")
print(" - any ga4_* column")
print(" - trend_direction, trend_pct, is_declining_label (label-derived, would leak)")
print(" - any data from the May 2026 target window")

Edge cases: just past the 90-day cutoff, still highly visible


,rank,content_hash_id,content_type,days_since_update,age_bucket,impressions,staleness_weight,action_score,reason_code,action



Edge cases: flagged TOO-FRESH but already highly visible


,rank,content_hash_id,content_type,days_since_update,age_bucket,impressions,staleness_weight,action_score,reason_code,action
24099,24100,content_65b8a4998e633d89,keyword article,22,1) 0-30d,8905.0,0.189922,1.727246,TOO-FRESH,monitor
24100,24101,content_d8ca9e67dbeeeb50,keyword article,26,1) 0-30d,8238.0,0.189922,1.712462,TOO-FRESH,monitor
24101,24102,content_80800697f291f292,keyword article,26,1) 0-30d,5988.0,0.189922,1.651885,TOO-FRESH,monitor
24361,24362,content_a41c59cf5fbb27e6,keyword article,26,1) 0-30d,4380.0,0.189922,1.592506,TOO-FRESH,monitor
24362,24363,content_55909550cb5e7b81,keyword article,22,1) 0-30d,2876.0,0.189922,1.512638,TOO-FRESH,monitor



=== LEAKAGE CHECK ===
Feature window used: March 2026 only (fact_content_daily_performance, month=2026-03)
Columns used from fact table: gsc_data_available, gsc_impressions
Columns used from dim_content: content_hash_id, content_type, content_updated_date, is_published, is_deleted

Columns explicitly NOT used anywhere in this pipeline:
 - gsc_clicks, gsc_avg_position (dropped with the CTR signal)
 - any ga4_* column
 - trend_direction, trend_pct, is_declining_label (label-derived, would leak)
 - any data from the May 2026 target window


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.